# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [6]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:1b'

In [7]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
openai = OpenAI()

API key looks good so far


In [8]:
# here is the question; type over this to ask something new
system_prompt ="You are a helpful programming sidekick."

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [9]:
# Get gpt-4o-mini to answer, with streaming
def get_tech_explaination(system_prompt, user_prompt):
  stream = openai.chat.completions.create(model=MODEL_GPT, messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question},
    ],
    stream=True,
    )
  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in stream:
      response += chunk.choices[0].delta.content or ''
      update_display(Markdown(response), display_id=display_handle.display_id)

In [11]:
get_tech_explaination(system_prompt, question)

This line of code uses a generator expression with the `yield from` statement to yield values from a set comprehension. Let's break it down step by step to understand what it does and its purpose.

1. **Generator Context**: The `yield from` statement is used within a generator function to yield all values from an iterable. The generator function would be defined using the `def` keyword and would typically look something like this:

    ```python
    def my_generator_function():
        yield from {book.get("author") for book in books if book.get("author")}
    ```

2. **Set Comprehension**: The part inside the curly braces `{}` is a set comprehension. It creates a set of unique authors from the collection of books. Here's what it does:
   - `for book in books`: Iterates over each `book` in the `books` collection (which is assumed to be a list or another iterable of dictionaries).
   - `book.get("author")`: Fetches the value associated with the key "author" from each `book` dictionary.
   - `if book.get("author")`: Filters out any `book` for which the "author" key does not exist or has a value of `None` or empty (i.e., only includes books that have a valid author).

3. **Resulting Set**: The set comprehension `{book.get("author") for book in books if book.get("author")}` will produce a set of unique author names. By using a set, duplicates will be automatically removed, meaning that if multiple books by the same author exist, that author will only appear once in the set.

4. **Yielding Authors**: The `yield from` statement then yields each author name from the set, one at a time. This means that the generator function will produce each author until all unique authors have been yielded.

### Why Use This Code:
- **Avoiding Duplicates**: Using a set automatically eliminates duplicate authors, which is often desired when collecting such data.
- **Generator**: The use of `yield from` allows for efficient iteration, enabling users to consume each author one at a time without needing to create a complete list of authors in memory.
- **Clarity**: The code is compact and clear. It expresses the intent to extract unique authors from a collection of books directly in one line, which can be more readable and maintainable.

### Example:
Assuming `books` looks something like this:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4", "author": None},
]
```

The `yield from` expression would yield "Author A" and "Author B" when consumed, as "Author A" would only be yielded once despite being present in multiple books.

In [10]:
# Get Llama 3.2 to answer
# Get gpt-4o-mini to answer, with streaming
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

def get_tech_explaination_llama(system_prompt, user_prompt):
  stream = ollama.chat.completions.create(model=MODEL_LLAMA, messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question},
    ],
    stream=True,
    )
  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in stream:
      response += chunk.choices[0].delta.content or ''
      update_display(Markdown(response), display_id=display_handle.display_id)

In [11]:
get_tech_explaination_llama(system_prompt, question)

Let's break down what this code does.

**Code Explanation**

This is a Python code snippet that uses the `yield from` syntax to create an iterator. Here's a step-by-step explanation:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

- We start with a list comprehension `{book.get("author") for book in books if book.get("author")}`.
  - This comprehension iterates over each `book` in the `books` list.

- The next line is using the `yield from` syntax, which effectively "forwards" the generator expression into another iterator. 

  Here's a more verbose way of writing this:

  ```python
generator_expression = (
    book.get("author")
    for book in books
    if book.get("author")
)
```

- `yield from` is similar to Python's `next()` function, but instead of returning the value, it "yields" the rest of the list. This means that once this entire line finishes being executed (i.e., when you execute `next(generator_expression)` or an explicit call to `for item in generator_expression`, any remaining values from the generator are yielded out.

  So, if there's one more value left after creating a full iteration, it gets added to another iterator. This process repeats until all items have been processed. 

- Finally, we add the list of authors that aren't null (`None`).

**Why `yield from`?**

This code seems unusual because in Python generators (also called " iterators" or "functoolsiterator"), you can't use a loop like this. When you define an iterator using `yield`, each call to `next()` yields one element and stops the iteration.

The reason for using `yield from` here is likely due to a specific requirement where we need to execute some operation (for example, creating a new list of authors) that happens after processing all the books. In this case, we've used a generator expression where the items are not actually values that will be part of our final output; they're parts of the process. `yield from` helps us ensure that all remaining items in the generator (in this case, articles' authors) aren't lost.

This code is useful when you need to keep processing or transforming some data until a certain point and then exit the process (e.g., fetching more data, performing an action before proceeding).